# Part 1

<h3>Load dataset and categorize columns</h3>

In [ ]:
import dask.dataframe as dd

df = dd.read_csv("data.csv")

categorical_columns = [
    "Label",
    "Traffic Type",
    "Traffic Subtype",
    "Protocol",
]

# List of columns to exclude
unnecessary_columns = [
    "Flow ID", "Timestamp", "Src IP", "Dst IP", "Src Port", "Dst Port"
]

binary_columns = [

    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
]

# Filter out the unnecessary ones
not_numerical_columns = categorical_columns + unnecessary_columns + binary_columns
numerical_columns = [col for col in df.columns.tolist() if col not in not_numerical_columns]
num_df = df[numerical_columns]

<h3>Compute Basic Stats</h3>

In [ ]:
stats = num_df.describe().compute()
# stats.to_csv("csv/stats/stats.csv")

In [ ]:
import dask.dataframe as dd

# correlation = num_df.corr().compute()
# dd.to_csv(correlation, "csv/stats/correlation.csv", single_file=True)
# del correlation
# gc.collect()

Fallback if correlation variable is lost

In [ ]:
import dask.dataframe as dd

correlation = dd.read_csv("csv/stats/correlation.csv").compute() 

# Get column names
col_names = correlation.columns.tolist()

# Create a new column with the column names as rows
correlation.insert(0, '', col_names)
correlation.set_index('', inplace=True)

In [ ]:
stats_correlation = stats.T.corr()

<h3>Print Basic Stats</h3>

In [ ]:
print(stats)

<h3>Plot Basic Stats</h3>

In [ ]:
import matplotlib.pyplot as plt

for idx, row in stats.iterrows():
    if idx == "count":
        continue
    plt.figure()
    row.plot(figsize=(20,10), kind='bar')

    plt.title(idx)
    plt.xlabel("Columns")
    plt.ylabel("Values")
    plt.tight_layout()
    plt.show()

<h2>Correlation between columns</h2>

<h3>Heatmap</h3>

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Mask upper triangle
mask = np.triu(np.ones_like(correlation, dtype=bool))

plt.figure(figsize=(20, 18))  # Increase size
sns.heatmap(correlation, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, annot=False, cbar_kws={"shrink": 0.5})
plt.title('Correlation Heatmap (Masked Upper Triangle)')
plt.show()

<h3>Print highest correlations</h3>

In [ ]:
high_corr_pairs = correlation.where(mask).stack()  # This converts the DataFrame to a Series with MultiIndex
high_corr_pairs = high_corr_pairs[high_corr_pairs > 0.75]  # Filter the correlations greater than 0.75

# Print the results
print("Correlation pairs")
for idx, value in high_corr_pairs.sort_values(ascending=False).items():
    if idx[0] == idx[1]:
        # Skip self-correlation
        continue
    
    print(f"{value:.2f} {idx[0]} - {idx[1]}")

<h2>Statistical Correlation</h2>

<h3>Heatmap</h3>

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Mask upper triangle
mask = np.triu(np.ones_like(stats_correlation, dtype=bool))

plt.figure(figsize=(20, 18))  # Increase size
sns.heatmap(stats_correlation, mask=mask, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, annot=False, cbar_kws={"shrink": 0.5})
plt.title('Correlation Heatmap (Masked Upper Triangle)')
plt.show()

<h3>Print</h3>

In [ ]:
print("Statistical Correlation pairs")
for idx, value in high_corr_pairs.sort_values(ascending=False).items():
    if idx[0] == idx[1]:
        # Skip self-correlation
        continue
    print(f"{value:.2f} {idx[0]} - {idx[1]}")

# Part 2

## Remove columns based on stats

In [ ]:
col_to_drop = []

### Get low variance features

In [ ]:
low_variance_features = stats.T[stats.T['std'] < 1e-6].index.tolist()
print("Low variance features:")
for feature in low_variance_features:
    print(feature)

### Drop features that include "Min"/"Max" and have high correlation with one that has "Mean" 

In [ ]:
to_remove = ["Min", "Max", "Avg"]
to_keep = ["Mean", "Std"]
for idx, value in high_corr_pairs.sort_values(ascending=False).items():
    if idx[0] == idx[1]:
        continue
    
    if any(rem in idx[0] for rem in to_remove) and any(keep in idx[1] for keep in to_keep):
        col_to_drop.append(idx[0])
    
    if any(rem in idx[1] for rem in to_remove) and any(keep in idx[0] for keep in to_keep):
        col_to_drop.append(idx[1])

### Drop columns

In [ ]:
col_to_drop.extend(low_variance_features)
col_to_drop.extend(unnecessary_columns)

col_to_drop = list(set(col_to_drop))

df_less_cols = df.drop(columns=col_to_drop)

### Combine Features

In [ ]:
# Συνενώσεις/συνδυασμοί χαρακτηριστικών

# Λόγος Fwd/Bwd Packet Count
df_less_cols["Fwd_Bwd_Packet_Ratio"] = df["Total Fwd Packet"] / (df["Total Bwd packets"] + 1e-6)

# Λόγος συνολικού μήκους δεδομένων
df_less_cols["Fwd_Bwd_Length_Ratio"] = df["Total Length of Fwd Packet"] / (df["Total Length of Bwd Packet"] + 1e-6)

# Λόγος μέσου μεγέθους πακέτου Fwd/Bwd
df_less_cols["Fwd_Bwd_PacketLengthMean_Ratio"] = df["Fwd Packet Length Mean"] / (df["Bwd Packet Length Mean"] + 1e-6)

# Μέση τιμή ρυθμού δεδομένων (bytes/s)
df_less_cols["Avg_Bytes_per_s"] = (df["Flow Bytes/s"] + df["Fwd Packets/s"] + df["Bwd Packets/s"]) / 3

# Λόγος ενεργής/ανενεργής περιόδου
df_less_cols["Active_Idle_Ratio"] = df["Active Mean"] / (df["Idle Mean"] + 1e-6)

# Λόγος αρχικού παραθύρου FWD/BWD
df_less_cols["Win_Init_Ratio"] = df["FWD Init Win Bytes"] / (df["Bwd Init Win Bytes"] + 1e-6)




columns_to_drop = [
    "Total Fwd Packet", "Total Bwd packets",
    "Total Length of Fwd Packet", "Total Length of Bwd Packet",
    "Fwd Packet Length Mean", "Bwd Packet Length Mean",
    "Flow Bytes/s", "Fwd Packets/s", "Bwd Packets/s",
    "Active Mean", "Idle Mean",
    "FWD Init Win Bytes", "Bwd Init Win Bytes"
]

# Αφαίρεση των στηλών από το DataFrame
df_less_cols = df_less_cols.drop(columns=columns_to_drop)

Seperate Malicious and Benign

In [ ]:
df_less_cols_benign = df_less_cols[df_less_cols["Label"] == "Benign"]
df_less_cols = df_less_cols[df_less_cols["Label"] == "Malicious"]

### Delete unecessary variables

In [ ]:
import gc

del df, num_df, stats, correlation, stats_correlation
gc.collect()

In [ ]:
# Export df_less_cols to CSV
df_less_cols.compute().to_csv("csv/compressed_datasets/df_less_cols.csv", index=False)
del df_less_cols
gc.collect()

In [ ]:
import dask.dataframe as dd

# Import it back as a DataFrame
df_less_cols = dd.read_csv("csv/compressed_datasets/df_less_cols.csv")

# Divide Benign and Malicious
df_less_cols_benign = df_less_cols[df_less_cols["Label"] == "Benign"].compute()
df_less_cols = df_less_cols[df_less_cols["Label"] == "Malicious"]

## Sampling

In [ ]:
sampled_df = df_less_cols.sample(frac=0.01, random_state=42).compute()

In [ ]:
import gc

sampled_df.to_csv("csv/compressed_datasets/sampled_df.csv", index=False)
del sampled_df
gc.collect()

## Clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
# Επιλογή χαρακτηριστικών για clustering
selected_features = [
    'Flow Duration',
    'Flow IAT Mean',
    'Fwd IAT Mean',
    'Bwd IAT Mean',
    'Packet Length Mean',
    'Average Packet Size',
    'Down/Up Ratio',
    'Subflow Fwd Packets',
    'Subflow Bwd Packets',
    'Fwd_Bwd_Packet_Ratio',
    'Avg_Bytes_per_s',
    'Active_Idle_Ratio'
]

features = df_less_cols[selected_features]

# Κλιμάκωση (scaling) των χαρακτηριστικών
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features.compute())

### K-Means

#### Elbow method using KMeans

In [ ]:
from sklearn.cluster import KMeans

inertia = []
for k in range(2, 11):
    print(f"k: {k}")
    kmeans = KMeans(n_clusters=k).fit(X_scaled)
    inertia.append(kmeans.inertia_)

#### Plot

In [ ]:
import matplotlib as plt

plt.plot(range(2, 11), inertia, marker='o')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

#### Start K-Means

In [ ]:
from sklearn.cluster import KMeans
# Step 1: Fit KMeans
kmeans = KMeans(n_clusters=10, random_state=42)
kmeans.fit(X_scaled)

#### Integrate cluster labels in the df

In [ ]:
# Compute cluster labels
cluster_labels = kmeans.labels_

# Add labels as a new column to a pandas dataframe (with same index as original df_less_cols)
df_compressed_kmeans = df_less_cols.compute()
df_compressed_kmeans['cluster'] = cluster_labels

#### Top-n farthest rows per cluster (diversity)

In [ ]:
from scipy.spatial.distance import cdist
import numpy as np

# Compute distances from each point to each centroid
distances = cdist(X_scaled, kmeans.cluster_centers_)
k = 100  # Number of farthest points per cluster
farthest_indices = []

for cluster_id in range(kmeans.n_clusters):
    # Mask for current cluster
    cluster_mask = kmeans.labels_ == cluster_id
    cluster_distances = distances[cluster_mask, cluster_id]
    cluster_indices = np.where(cluster_mask)[0]

    # Sort distances descending and pick top-k
    k_farthest = cluster_indices[np.argsort(cluster_distances)[-k:]]
    
    farthest_indices.extend(k_farthest)

df_compressed_kmeans = df_compressed_kmeans.iloc[farthest_indices].reset_index(drop=True)

#### Balance dataset

In [ ]:
# Compute df_less_cols_benign and concatenate to df_compressed_kmeans
import pandas as pd
df_compressed_kmeans = pd.concat([df_compressed_kmeans, df_less_cols_benign], ignore_index=True)

In [ ]:
import gc
# Export df_less_cols to CSV
df_compressed_kmeans.to_csv("csv/compressed_datasets/df_compressed_kmeans_balanced_farthest_k20.csv", index=False)
del df_compressed_kmeans
gc.collect()

### DBSCAN

#### DBSCAN Parameters

In [ ]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import matplotlib.pyplot as plt

# Choose k as n_neighbors (usually min_samples for DBSCAN)
k = 108

# Compute the nearest neighbors
neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(features.compute())

# Get distances to the k nearest neighbors
distances, indices = neighbors_fit.kneighbors(features.compute())

# Sort the distances to the k-th neighbor (use k-1 because index starts at 0)
k_distances = np.sort(distances[:, k-1])

# Plot the k-distance graph
plt.figure(figsize=(8, 4))
plt.plot(k_distances)
plt.xlabel("Points sorted by distance to {}th nearest neighbor".format(k))
plt.ylabel("Distance")
plt.title("k-distance Graph for DBSCAN (k = {})".format(k))
plt.grid(True)
plt.show()

In [ ]:
from kneed import KneeLocator
import matplotlib.pyplot as plt

kneedle = KneeLocator(range(len(k_distances)), k_distances, curve='convex', direction='increasing')
print(f"Suggested epsilon: {kneedle.knee_y}")
plt.axhline(y=kneedle.knee_y, color='r', linestyle='--')

#### DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=2.5e-7, min_samples=55)
dbscan.fit_predict(X_scaled)

# Part 3

## Neural Network (Classification on "Label")

### Compressed Datasets

In [ ]:
import pandas as pd
df = pd.read_csv("csv/compressed_datasets/df_compressed_kmeans_balanced_farthest.csv")

# Drop columns that are not needed
# Traffic Type and Traffic Subtype were dropped for possible data leakage
df = df.drop(columns=["Traffic Type", "Traffic Subtype", "cluster"])

In [ ]:
import pandas as pd

df = pd.read_csv("csv/compressed_datasets/sampled_df.csv")

# Drop columns that are not needed
# Traffic Type and Traffic Subtype were dropped for possible data leakage
df = df.drop(columns=["Traffic Type", "Traffic Subtype"])

### Preprocessing

#### Column categorization

In [ ]:
classification_columns = [
    "Label"
]
categorical_columns = [
    "Protocol",
]

binary_columns = [
    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
]

# Filter out the unnecessary ones
not_numerical_columns = categorical_columns + classification_columns + binary_columns
numerical_columns = [col for col in df.columns.tolist() if col not in not_numerical_columns]

#### Preprocessing Pipelines

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Preprocessing pipelines
standard_scaler = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(transformers=[
    ('num', standard_scaler, numerical_columns),
    ('cat', categorical_transformer, categorical_columns),
], remainder='passthrough')  # leaves binary columns as they are



# Separate input and output
X = df.drop(columns=["Label"])  # Adjust target if needed
Y = df['Label'].map({'Benign': 0, 'Malicious': 1})

X = preprocessor.fit_transform(X)

### Hold out test set

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming X and y are numpy arrays or pandas
X_temp, X_test, Y_temp, Y_test = train_test_split(
    X, Y, test_size=0.1, random_state=42, stratify=Y  # keep class balance
)

### Learning parameters

In [ ]:
from sklearn.model_selection import StratifiedKFold
from keras.callbacks import EarlyStopping # type: ignore

kfold = StratifiedKFold(n_splits=5, shuffle=True)


hidden_layer_activation = 'relu'
output_activation = 'sigmoid'

I = X.shape[1]  # Number of input features
H = I  # Number of hidden units

epochs = 100
batch_size = 100


early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,           # stop if val_loss doesn't improve for 2 epochs
    restore_best_weights=True  # bring back the best model
)

In [ ]:
from keras.models import Sequential # type: ignore
from keras.layers import Dense, Input # type: ignore
from keras.regularizers import l2 # type: ignore
import matplotlib.pyplot as plt
import numpy as np

avg_loss_per_epoch = np.zeros(epochs)
avg_val_loss_per_epoch = np.zeros(epochs)

avg_mse_per_epoch = np.zeros(epochs)
avg_val_mse_per_epoch = np.zeros(epochs)

avg_acc_per_epoch = np.zeros(epochs)
avg_val_acc_per_epoch = np.zeros(epochs)

lossList = []
accuracyList = []

mseList=[]
valMseList=[]

valLossList = []
valAccuracyList = []

# Track metrics per fold
loss_per_fold = []
val_loss_per_fold = []

mse_per_fold = []
val_mse_per_fold = []

acc_per_fold = []
val_acc_per_fold = []

min_epochs = float('inf')  # To determine the shortest run

for j, (train_idx, val_idx) in enumerate(kfold.split(X_temp, Y_temp)):
    X_train, Y_train = X_temp[train_idx], Y_temp.iloc[train_idx]
    X_val, Y_val = X_temp[val_idx], Y_temp.iloc[val_idx]

    model = Sequential()
    model.add(Input(shape=(X.shape[1],)))
    model.add(Dense(X.shape[1], activation=hidden_layer_activation, kernel_regularizer=l2(0.001)))  # Add regularizer here
    model.add(Dense(1, activation=output_activation))

    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['mse', 'accuracy'])

    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[early_stop]
    )

    n_epochs = len(history.history['loss'])
    min_epochs = min(min_epochs, n_epochs)

    loss_per_fold.append(history.history['loss'])
    val_loss_per_fold.append(history.history['val_loss'])
    
    mse_per_fold.append(history.history['mse'])
    val_mse_per_fold.append(history.history['val_mse'])
    
    acc_per_fold.append(history.history['accuracy'])
    val_acc_per_fold.append(history.history['val_accuracy'])

    # Evaluate on held-out test set
    score = model.evaluate(X_test, Y_test, verbose=0)
    lossList.append(score[0])
    mseList.append(score[1])
    accuracyList.append(score[2])
    
    valLossList.append(np.mean(history.history['val_loss']))
    valMseList.append(np.mean(history.history['val_mse']))
    valAccuracyList.append(np.mean(history.history['val_accuracy']))

    print(f"Fold {j+1}: Test Loss = {score[0]:.4f}, Test MSE = {score[1]:.4f}, Test MSE = {score[2]:.4f}")

# Average metrics up to min_epochs
avg_loss_per_epoch = np.mean([loss[:min_epochs] for loss in loss_per_fold], axis=0)
avg_val_loss_per_epoch = np.mean([val_loss[:min_epochs] for val_loss in val_loss_per_fold], axis=0)

avg_mse_per_epoch = np.mean([mse[:min_epochs] for mse in mse_per_fold], axis=0)
avg_val_mse_per_epoch = np.mean([val_mse[:min_epochs] for val_mse in val_mse_per_fold], axis=0)

avg_acc_per_epoch = np.mean([acc[:min_epochs] for acc in acc_per_fold], axis=0)
avg_val_acc_per_epoch = np.mean([val_acc[:min_epochs] for val_acc in val_acc_per_fold], axis=0)

# Final report
print(f"\nAverage Test Loss: {np.mean(lossList):.4f}")
print(f"Average Test Accuracy: {np.mean(accuracyList):.4f}")
print(f"Average Test MSE: {np.mean(mseList):.4f}")
print(f"Average Validation Loss: {np.mean(valLossList):.4f}")
print(f"Average Validation Accuracy: {np.mean(valAccuracyList):.4f}")
print(f"Average Validation MSE: {np.mean(valMseList):.4f}")

# Plotting
plt.figure(figsize=(15, 5))

# Loss Plot
plt.subplot(1, 3, 1)
plt.plot(range(1, min_epochs + 1), avg_loss_per_epoch, linestyle='-', color='r')
plt.plot(range(1, min_epochs + 1), avg_val_loss_per_epoch, linestyle='--', color='b')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Average Loss per Epoch")
plt.legend(['Training Loss', 'Validation Loss'])
plt.grid()

# MSE Plot
plt.subplot(1, 3, 2)
plt.plot(range(1, min_epochs + 1), avg_mse_per_epoch, linestyle='-', color='r')
plt.plot(range(1, min_epochs + 1), avg_val_mse_per_epoch, linestyle='--', color='b')
plt.xlabel("Epochs")
plt.ylabel("MSE")
plt.title("Average MSE per Epoch")
plt.legend(['Training MSE', 'Validation MSE'])
plt.grid()

# Accuracy Plot
plt.subplot(1, 3, 3)
plt.plot(range(1, min_epochs + 1), avg_acc_per_epoch, linestyle='-', color='b')
plt.plot(range(1, min_epochs + 1), avg_val_acc_per_epoch, linestyle='--', color='r')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Average Accuracy per Epoch")
plt.legend(['Training Accuracy', 'Validation Accuracy'])
plt.grid()

plt.show()

## SVM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    roc_curve,
    auc,
)
classification_columns = [
    "Label"
]
categorical_columns = [
    "Protocol",
]

binary_columns = [
    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
]

# Filter out the unnecessary ones
not_numerical_columns = categorical_columns + classification_columns + binary_columns
numerical_columns = [col for col in df.columns.tolist() if col not in not_numerical_columns]


# Preprocessing pipelines
standard_scaler = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(transformers=[
    ('num', standard_scaler, numerical_columns),
    ('cat', categorical_transformer, categorical_columns),
], remainder='passthrough')  # leaves binary columns as they are



# Separate input and output
X = df.drop(columns=["Label"])  # Adjust target if needed
Y = df['Label'].map({'Benign': 0, 'Malicious': 1})

X = preprocessor.fit_transform(X)


# --- Split into training and test sets ---
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.1, stratify=Y, random_state=42
)

# --- 3. Build SVM model with linear kernel and standardization ---
model = make_pipeline(
    SVC(kernel='linear', C=1, probability=True)
)

# --- 4. Train the model ---
model.fit(X_train, y_train)

# --- 5. Predict and Evaluate ---
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # Probabilities for positive class

# --- 6. Print evaluation metrics ---
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

# --- 7. Plot ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='blue', lw=2, label=f"ROC curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

# --- 8. Perform 3-Fold Stratified Cross-Validation (F1 Score) ---
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
f1_scores = cross_val_score(model, X, Y, cv=cv, scoring='f1')

print("\n3-Fold Cross-Validation F1 Scores:", f1_scores)
print("Mean F1 Score:", f1_scores.mean())

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Apply PCA to reduce to 2D for visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Plot
plt.figure(figsize=(7, 6))
plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=Y, cmap='coolwarm', alpha=0.6,
    edgecolors='k'
)
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('PCA Projection Colored by Label')
plt.grid(True)
plt.colorbar(label='Label (0=Benign, 1=Malicious)')
plt.show()

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X)

plt.figure(figsize=(7, 6))
plt.scatter(
    X_tsne[:, 0], X_tsne[:, 1],
    c=Y, cmap='coolwarm', alpha=0.6,
    edgecolors='k'
)
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('t-SNE Projection Colored by Label')
plt.grid(True)
plt.colorbar(label='Label (0=Benign, 1=Malicious)')
plt.show()

In [ ]:
df_corr = df.copy()
df_corr['Label_Binary'] = df_corr['Label'].map({'Benign': 0, 'Malicious': 1})

# Drop non-numeric columns
df_corr_numeric = df_corr.select_dtypes(include='number')

# Correlation with the label
label_corr = df_corr_numeric.corr()['Label_Binary'].sort_values(ascending=False)

print("Top positively correlated features:\n", label_corr.head(15))
print("\nTop negatively correlated features:\n", label_corr.tail(15))

In [ ]:
# Assuming your X_train and X_test are numpy arrays or DataFrames
import pandas as pd

X_train_df = pd.DataFrame(X_train)
X_test_df = pd.DataFrame(X_test)

# Find common rows
duplicates = pd.merge(X_test_df, X_train_df, how='inner')
print(f"Number of overlapping rows between train and test: {len(duplicates)}")

In [ ]:
duplicate_flows = df.duplicated().sum()
print(f"Exact duplicates in the full dataset: {duplicate_flows}")

## Neural Network (Classification on "Traffic Type)

### Compressed Datasets

In [ ]:
import pandas as pd
df = pd.read_csv("csv/compressed_datasets/df_compressed_kmeans_balanced_farthest.csv")

# Drop columns that are not needed
# Traffic Type and Traffic Subtype were dropped for possible data leakage
df = df.drop(columns=["Label", "Traffic Subtype", "cluster"])

In [ ]:
import pandas as pd

df = pd.read_csv("csv/compressed_datasets/sampled_df.csv")

# Drop columns that are not needed
# Traffic Type and Traffic Subtype were dropped for possible data leakage
df = df.drop(columns=["Label", "Traffic Subtype"])

### Preprocessing

#### Column categorization

In [ ]:
classification_columns = [
    "Traffic Type",
]
categorical_columns = [
    "Protocol",
]

binary_columns = [
    "Fwd PSH Flags",
    "Bwd PSH Flags",
    "Fwd URG Flags",
    "Bwd URG Flags",
]

# Filter out the unnecessary ones
not_numerical_columns = categorical_columns + classification_columns + binary_columns
numerical_columns = [col for col in df.columns.tolist() if col not in not_numerical_columns]

#### Preprocessing Pipelines

In [ ]:
# Preprocessing pipelines
standard_scaler = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine transformers
preprocessor = ColumnTransformer(transformers=[
    ('num', standard_scaler, numerical_columns),
    ('cat', categorical_transformer, categorical_columns),
], remainder='passthrough')  # leaves binary columns as they are



# Separate input and output
X = df.drop(columns=["Traffic Type"])  # Adjust target if needed
Y = df['Traffic Type']

X = preprocessor.fit_transform(X)

# Initialize encoder
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

# Fit and transform Y
Y = encoder.fit_transform(Y)

### Hold out test set

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming X and y are numpy arrays or pandas
X_temp, X_test, Y_temp, Y_test = train_test_split(
    X, Y, test_size=0.1, random_state=42, stratify=Y  # keep class balance
)

### Learning parameters

In [ ]:
from sklearn.model_selection import StratifiedKFold
from keras.callbacks import EarlyStopping # type: ignore

kfold = StratifiedKFold(n_splits=5, shuffle=True)


hidden_layer_activation = 'relu'
output_activation = 'softmax'

I = X.shape[1]  # Number of input features
H = I  # Number of hidden units

epochs = 100
batch_size = 100


early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,           # stop if val_loss doesn't improve for 2 epochs
    restore_best_weights=True  # bring back the best model
)

In [ ]:
from keras.models import Sequential # type: ignore
from keras.layers import Dense, Input # type: ignore
from keras.regularizers import l2 # type: ignore
import matplotlib.pyplot as plt
import numpy as np

avg_loss_per_epoch = np.zeros(epochs)
avg_val_loss_per_epoch = np.zeros(epochs)

avg_mse_per_epoch = np.zeros(epochs)
avg_val_mse_per_epoch = np.zeros(epochs)

avg_acc_per_epoch = np.zeros(epochs)
avg_val_acc_per_epoch = np.zeros(epochs)

lossList = []
accuracyList = []

valLossList = []
valAccuracyList = []

# Track metrics per fold
loss_per_fold = []
val_loss_per_fold = []
acc_per_fold = []
val_acc_per_fold = []

min_epochs = float('inf')  # To determine the shortest run

for j, (train_idx, val_idx) in enumerate(kfold.split(X_temp, Y_temp)):
    X_train, Y_train = X_temp[train_idx], Y_temp[train_idx]
    X_val, Y_val     = X_temp[val_idx], Y_temp[val_idx]

    model = Sequential()
    model.add(Input(shape=(X.shape[1],)))
    model.add(Dense(X.shape[1], activation='relu', kernel_regularizer=l2(0.001)))  # Add regularizer here
    model.add(Dense(8, activation='softmax'))

    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=0,
        callbacks=[early_stop]
    )

    n_epochs = len(history.history['loss'])
    min_epochs = min(min_epochs, n_epochs)

    loss_per_fold.append(history.history['loss'])
    val_loss_per_fold.append(history.history['val_loss'])
    acc_per_fold.append(history.history['accuracy'])
    val_acc_per_fold.append(history.history['val_accuracy'])

    # Evaluate on held-out test set
    score = model.evaluate(X_test, Y_test, verbose=0)
    lossList.append(score[0])
    accuracyList.append(score[1])
    
    valLossList.append(np.mean(history.history['val_loss']))
    valAccuracyList.append(np.mean(history.history['val_accuracy']))

    print(f"Fold {j+1}: Test Loss = {score[0]:.4f}, Test Accuracy = {score[1]:.4f}")

# Average metrics up to min_epochs
avg_loss_per_epoch = np.mean([loss[:min_epochs] for loss in loss_per_fold], axis=0)
avg_val_loss_per_epoch = np.mean([val_loss[:min_epochs] for val_loss in val_loss_per_fold], axis=0)
avg_acc_per_epoch = np.mean([acc[:min_epochs] for acc in acc_per_fold], axis=0)
avg_val_acc_per_epoch = np.mean([val_acc[:min_epochs] for val_acc in val_acc_per_fold], axis=0)

# Final report
print(f"\nAverage Test Loss: {np.mean(lossList):.4f}")
print(f"Average Test Accuracy: {np.mean(accuracyList):.4f}")
print(f"Average Validation Loss: {np.mean(valLossList):.4f}")
print(f"Average Validation Accuracy: {np.mean(valAccuracyList):.4f}")

# Plotting
plt.figure(figsize=(15, 5))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(range(1, min_epochs + 1), avg_loss_per_epoch, linestyle='-', color='r')
plt.plot(range(1, min_epochs + 1), avg_val_loss_per_epoch, linestyle='--', color='b')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Average Loss per Epoch")
plt.legend(['Training Loss', 'Validation Loss'])
plt.grid()

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot(range(1, min_epochs + 1), avg_acc_per_epoch, linestyle='-', color='b')
plt.plot(range(1, min_epochs + 1), avg_val_acc_per_epoch, linestyle='--', color='r')
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Average Accuracy per Epoch")
plt.legend(['Training Accuracy', 'Validation Accuracy'])
plt.grid()

plt.show()